# Lab 4: 양자 우위를 향하여

# Chapter 4: 변분 문제: Sample-based Quantum Diagonalization

### *참고: Lab4c는 배지 취득에 필수가 아닌 보너스 랩입니다*

*보너스 파트를 제외한 이 노트북의 QPU (Quantum Processing Unit, 양자 처리 장치) 시간 사용 예상치: Nighthawk r1에서 10초 또는 Heron r2에서 2초. (참고: 이는 예상치일 뿐이며, 실제 실행 시간은 다를 수 있습니다.)*

*QPU 사용량 예상치는 backend 실행 시간만을 반영합니다. 큐 대기, 캘리브레이션, 런타임 session 지연으로 인해 QPU 시간이 몇 초에 불과한 작업도 전체 실행 시간은 수 분 이상으로 늘어날 수 있습니다.*

### 목차

- Exercise 1: LUCJ 회로를 Nighthawk 디바이스에 매핑하기
- Exercise 2: 결과 bitstring 살펴보기
- Exercise 3: bitstring 복원하기
- Exercise 4: Classical reference subspace로 SQD 보강하기
- Scored Exercise: 최적의 부분공간 찾기!

Lab 4a에서 양자 우위가 무엇을 의미하는지, 그리고 [Quantum Advantage Tracker](https://quantum-advantage-tracker.github.io/)가 이를 세 가지 범주 — 고전적으로 검증 가능한 문제, 변분 문제, 관측가능량 측정 — 로 어떻게 분류하는지 살펴보았습니다. $ \mathrm{Fe}_4\mathrm{S}_4$ 분자 계산(변분)과 Loschmidt Echo 계산(관측가능량 측정)과 같은 예시도 확인했습니다.

이제 두 번째 범주인 **변분 문제**를 살펴보겠습니다. 이는 고급 시뮬레이션을 위해 바닥 상태를 찾는 화학 및 재료 과학 분야의 문제입니다. 재료의 거동은 양자역학으로 직접 기술되므로, 양자 계산을 적용하기에 이상적인 분야임은 당연합니다.

이 랩에서는 실제 양자 컴퓨터에서 **Sample-based Quantum Diagonalization (SQD, 샘플 기반 양자 대각화)** 를 사용하여 질소 분자(N₂)의 **바닥 상태 에너지**를 측정합니다.

SQD는 하이브리드 양자-고전 방법입니다. 양자 컴퓨터는 파라미터화된 회로를 실행하여 어떤 전자 배열이 가장 중요한지 *제안*하고, 고전 컴퓨터는 그렇게 선별된 작은 구성 집합에서 분자 Hamiltonian을 대각화하는 선형 대수 작업을 수행합니다.

이 챕터를 마치면, 화학 회로를 두 가지 IBM 디바이스 구조에 매핑하고, 잡음이 있는 하드웨어 bitstring을 정제하며, 실제 양자 하드웨어를 사용하여 고전적 brute-force 방법을 능가할 수 있습니다.

### 필요한 라이브러리 설치

> **Windows 사용자 참고:**
> 필요한 라이브러리 중 하나인 pyscf는 Windows OS를 공식적으로 지원하지 않습니다. 이 랩에서는 [qBraid](https://qbraid.com/)나 [Google Colab](https://colab.research.google.com/) 사용을 고려해 주세요. 추가적인 환경 설정을 할 수 있으시다면, [Windows Subsystem for Linux](https://learn.microsoft.com/en-us/windows/wsl/install)를 통해 로컬 환경을 온전히 활용할 수 있습니다.

In [ ]:
%pip install qiskit-addon-sqd
%pip install ffsim

### 임포트

In [ ]:
# 필요한 패키지 임포트
import warnings
warnings.filterwarnings("ignore")

import copy
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import cast
from itertools import combinations
from functools import partial

import pyscf
import pyscf.cc
import pyscf.mcscf
import ffsim

from qiskit.circuit import QuantumCircuit, QuantumRegister, CircuitInstruction, Barrier
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_error_map
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit_ibm_runtime.fake_provider import FakeMiami

from qiskit_addon_sqd.fermion import SCIResult, solve_sci_batch
from qiskit_addon_sqd.subsampling import subsample
from qiskit_addon_sqd.counts import bit_array_to_arrays, bitstring_matrix_to_integers

In [ ]:
from qc_grader.challenges.qgss_2026 import (
    check_progress,
    grade_lab4c_ex1a,
    grade_lab4c_ex1b,
    grade_lab4c_ex2a,
    grade_lab4c_ex2b,
    grade_lab4c_ex3a,
    grade_lab4c_ex3b,
    grade_lab4c_ex4,
    grade_lab4c_exbonus,
)

랩 전반에 걸쳐 `check_progress` 함수를 사용하여 완료한 연습 문제 수를 확인할 수 있습니다:

In [ ]:
check_progress()

## 배경: 5분 만에 배우는 분자 양자 화학

이 랩에 화학 배경 지식은 **필요하지 않습니다**. 이 섹션은 이 랩에 필요한 최소한의 배경 지식을 제공합니다.

### 분자, 오비탈, 전자

분자를 작은 콘서트 홀이라고 생각하세요.
**분자 오비탈**은 그 홀의 좌석입니다.
**전자**는 한 가지 엄격한 규칙을 따르는 손님입니다. 각 좌석(오비탈)에는 주어진 스핀의 손님이 *최대 한 명만* 앉을 수 있습니다 — 이것이 파울리 배타 원리입니다.

각 공간 오비탈은 두 전자를 수용할 수 있으며, 하나는 **spin-α (↑)**, 하나는 **spin-β (↓)** 입니다.
두 스핀 종류를 항상 별도로 추적하는데, 이것이 바로 나중에 측정하는 bitstring이 α 절반과 β 절반으로 나뉘는 이유입니다.

### 바닥 상태와 그 중요성

**바닥 상태**는 총 에너지가 가장 낮은 배치입니다.
화학자들에게 분자가 안정적인지, 결합 강도는 어느 정도인지, 반응 여부를 알려줍니다.
바닥 상태 에너지의 정확한 계산은 양자 컴퓨터의 가장 매력적인 응용 분야 중 하나입니다. 가능한 배치의 수가 *지수적으로* 증가하기 때문입니다 — $n$개의 오비탈과 $N_e$개의 전자에 대해 스핀 종류당 $\binom{n}{N_e}$가지 배열이 존재하며, 이는 어떤 고전 컴퓨터의 능력도 쉽게 초과합니다.
이 랩의 N₂의 경우, 26개의 active 오비탈과 스핀당 5개의 전자를 가지며, 이미 스핀 섹터당 $\binom{26}{5} = 65{,}780$가지 구성이 존재합니다 — 전체 Hilbert space 차원은 두 스핀 섹터의 곱이므로 총 40억 개 이상의 상태가 가능합니다.

바닥 상태 찾기는 수학적으로는 **고윳값 문제**로 표현됩니다:

$$\hat{H} |\psi_0\rangle = E_0 |\psi_0\rangle$$

여기서 $|\psi_0\rangle$은 바닥 상태 파동함수이고 $E_0$는 가장 낮은 에너지 고윳값입니다.
동등하게, **변분 원리** 는 이를 최소화 문제로 표현합니다:

$$E_0 = \min_{\|\psi\|=1} \langle \psi | \hat{H} | \psi \rangle$$

임의의 양자 상태 $|\psi\rangle$은 $E_0$에 대한 **상한**을 제공합니다: $\langle\psi|\hat{H}|\psi\rangle \geq E_0$.
이 부등식은 SQD를 포함한 모든 변분 양자 알고리즘의 기반이 됩니다.

이차 양자화에서 해밀토니안은 1체 및 2체 항으로 분리됩니다:

$$\hat{H} = \sum_{pq} h_{pq}\,\hat{a}^\dagger_p \hat{a}_q + \frac{1}{2}\sum_{pqrs} g_{pqrs}\,\hat{a}^\dagger_p \hat{a}^\dagger_q \hat{a}_s \hat{a}_r + E_\text{nuc}$$

- $h_{pq}$: **1전자 적분** — 운동 에너지와 전자-핵 인력 (코드에서 `hcore`).
- $g_{pqrs}$: **2전자 반발 적분** — 전자-전자 쿨롱 반발 (코드에서 `eri`).
- $E_\text{nuc}$: 상수 핵 반발 에너지.

SQD는 $\hat{H}$를 작은 **부분공간** — 신중하게 선택된 Slater determinant 집합 — 에 투영하고 그 부분공간 내에서 정확히 대각화하여 이 문제를 풉니다.
양자 컴퓨터의 역할은 어떤 determinant가 그 부분공간에 속하는지 제안하는 것입니다.

### 고전값의 레퍼런스

이 랩에서는 세 가지의 고전적인 레퍼런스 값이 등장합니다:

| 방법 | 역할 | 정확도 |
|---|---|---|
| **Hartree-Fock (HF)** | 각 전자가 다른 모든 전자를 무시하고 가장 저렴한 좌석을 독립적으로 선택합니다. 빠르지만, 전자들이 서로를 피하는 경향인 전자 *상관관계*를 무시합니다. | $E_0$의 상한; 대략적인 측정값. |
| **CCSD** (Coupled Cluster Singles & Doubles) | HF에서 출발하여, 대수적 진폭 $t_1$ (singles)와 $t_2$ (doubles)를 사용해 HF 배열에서 1- 및 2-전자 "점프"를 추가합니다. 대부분의 상관관계를 포착합니다. | $E_0$의 효율적이고 합리적인 고전적 측정값을 제공합니다. |
| **Classical selected-CI** (이 랩의 reference subspace) | 작고 직접 선택된 Slater determinant 부분공간에서 Hamiltonian을 구성하고 대각화합니다. 고품질 determinant가 제공되면 거의 정확한 수준에 근접할 수 있습니다. 이 랩의 reference subspace은 brute-force 방법으로 이를 선택합니다: 오비탈을 특정한 선택 기준 없이 순서대로 열거하여 대각화합니다. SQD 결과가 이 기준선을 능가하면, *양자 유용성*을 입증한 것입니다: 양자 샘플러가 brute-force 열거법으는 찾을 수 없는 구성을 찾아낸 것입니다. | 부분공간의 품질에 따라 달라집니다. 랩의 brute-force 버전은 일반적으로 HF보다 좋지만 CCSD에는 미치지 못합니다. |

> **핵심 포인트:** CCSD를 양자 회로를 초기화하는 데 *활용*하기도 합니다 — 그 진폭은 전자들이 오비탈 사이에서 어떻게 이동하는지를 기술하며, 그 화학적 직관을 양자 ansatz의 파라미터에 직접 이식합니다.

## 1. N₂ 분자와 해밀토니안 구성

### 질소 분자

이 랩에서는 **분자 질소(N₂)** 를 연구합니다: 2.0 Å 떨어진 두 질소 원자.
이는 N₂의 평형 결합 길이(~1.1 Å)보다 약 두 배 늘어난 구조입니다 — 전자 상관관계 효과가 더 두드러지고, HF와 정확한 바닥 상태 사이의 간격이 더 커져 양자 컴퓨팅 응용이 더 설득력 있는 기하 구조입니다.
분자 기술에는 **`cc-pvdz` basis set** 을 사용합니다 — 정확도와 계산 비용의 균형이 잘 잡힌 표준 basis set입니다.
이 basis에서 Hartree-Fock을 실행하면 총 28개의 공간 오비탈이 생깁니다.

### Active space와 동결된 오비탈

모든 오비탈이 동등하게 중요하지는 않습니다.
가장 안쪽의 **코어** 오비탈 (각 질소 원자의 1s 전자)은 에너지적으로 깊은 곳에 위치하며 본질적으로 절대 변하지 않습니다 — 분자가 무엇을 하든 완벽히 만족스러운 상태입니다.
이 오비탈들을 양자 역학적으로 계산하면 귀중한 큐비트를 낭비하게 됩니다.

따라서 이들을 **동결** 합니다:
- `n_frozen = 2`는 양자 처리에서 가장 낮은 에너지의 코어 오비탈 2개를 제거합니다.
- `active_space = range(n_frozen, mol.nao_nr())`는 화학적으로 흥미로운 활동 — 결합 형성, 전자 상관관계 — 이 실제로 일어나는 오비탈 집합입니다.

이것이 **active-space 근사** 입니다: 전체에 대해 완전한 HF를 풀고, 그다음 active-space 전자만 양자 컴퓨터에서 처리합니다.
동결 후에 남는 값은 다음과 같습니다:
- `num_orbitals` 개의 active spatial 오비탈 → 스핀 섹터당 큐비트 수와 동일합니다.
- `(num_elec_a, num_elec_b)` 개의 spin-α와 spin-β active 전자.  N₂는 closed-shell singlet이므로 `num_elec_a == num_elec_b`.

아래 코드 셀은 **1전자 적분** (`hcore`), **2전자 반발 적분** (`eri`), 상수 **핵 반발 에너지**를 추출합니다 — 이들은 active space로 제한된 $\hat{H}$의 수치적 표현으로, 고전 eigensolver가 나중에 대각화에 활용합니다.

In [ ]:
# N2 분자 구성
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (2.0, 0, 0)]],
    basis="cc-pvdz",
)

# Active space 정의
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# 분자 적분 가져오기
scf = pyscf.scf.RHF(mol).run()
num_orbitals = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2
cas = pyscf.mcscf.CASCI(scf, num_orbitals, (num_elec_a, num_elec_b))
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

# 분자 정보 출력
print(f"Number of molecular orbitals: {num_orbitals}")
print(f"Number of electrons (a, b): {(num_elec_a, num_elec_b)}")

### 양자 ansatz를 초기화하기 위한 CCSD 실행

양자 회로를 구성하기 전에, **CCSD**를 고전적으로 실행합니다.
CCSD를 최종 답으로 사용하는 것이 *아닙니다* — 그렇게 하면 본 목적에서 벗어납니다.
여기서는 CCSD의 **진폭을 추출**합니다:

- `t1[i, a]` — 하나의 전자가 점유된 오비탈 $i$에서 가상 오비탈 $a$로 "점프"하는 진폭.
- `t2[i, j, a, b]` — 전자 *쌍*이 $(i, j)$에서 $(a, b)$로 산란되는 진폭.

이 진폭들은 전자들이 어떻게 상관관계를 맺는지에 대한 CCSD의 화학적 직관을 담고 있습니다.
다음 섹션에서 `ffsim` 라이브러리가 이 진폭들을 읽고 양자 회로의 회전 각도로 컴파일하여, 사실상 고전 화학 지식을 고품질의 초기 파라미터로 양자 디바이스에 이식합니다.

In [ ]:
# ansatz 초기화를 위한 CCSD t2 진폭 가져오기
ccsd = pyscf.cc.CCSD(scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]).run(max_cycle=1000)
t1 = ccsd.t1
t2 = ccsd.t2

## 2. 양자 상태 준비: LUCJ ansatz

### LUCJ ansatz란?

양자 상태 준비 단계의 목표는 N₂ 바닥 상태를 잘 근사하는 양자 회로를 구성하는 것입니다. 이 회로를 샘플링하면 실제 바닥 상태에 가까운 전자 배치를 얻을 수 있습니다.
여기서는 **LUCJ ansatz** — **Local Unitary Cluster Jastrow** — 를 사용합니다. 이는 두 가지 요소를 통해 전자 상관관계를 포착하는 하드웨어 효율적인 매개변수화 회로입니다:

1. **오비탈 회전** $e^{i\hat{K}}$: 분자 오비탈을 혼합하는 단일체 유니터리 (좌석을 재배치하는 것으로 생각하면 됩니다).
2. **Jastrow factor** $e^{i\sum_{pq} J_{pq}\,\hat{n}_p \hat{n}_q}$: 오비탈 $p$와 $q$를 동시에 점유하는 전자 쌍에 패널티 또는 보상을 주는 대각 상호작용 (전자들이 서로의 존재에 반응합니다).

하나의 UCJ (Unitary Cluster Jastrow) 층은 다음과 같은 형태입니다:

$$U_\mu = e^{i\hat{K}_\mu}\; e^{i\sum_{p<q} J^{(\mu)}_{pq}\,\hat{n}_p \hat{n}_q}\; e^{-i\hat{K}_\mu}$$

여기서 $\hat{n}_p = a^\dagger_p a_p$는 오비탈 $p$의 점유수 연산자입니다.
$a^\dagger_p$는 오비탈 $p$에 전자를 **생성**하고 $a_p$는 전자를 **소멸**시킵니다 — 이것이 다체 양자 시스템의 표준 **이차 양자화** 언어입니다.
$\hat{n}_p$는 단순히 점유 상태에 대한 사영자로, 오비탈 $p$가 점유되어 있으면 1, 비어 있으면 0입니다.
전체 ansatz는 `n_reps`개의 레이어를 사용합니다: $U = U_{n_\text{reps}} \cdots U_1$.
여기서는 `n_reps = 1`을 사용합니다.

### CCSD 진폭에서 회로 매개변수로

`ffsim.UCJOpSpinBalanced.from_t_amplitudes(t1, t2, n_reps, interaction_pairs)`는 CCSD의 단일 및 이중 여기 진폭을 읽어 회전 각도 $K_\mu$와 Jastrow 결합 $J_{pq}^{(\mu)}$로 컴파일합니다.
쉽게 말해, CCSD가 "오비탈 $i$와 $a$가 강하게 교환된다"고 하면, $e^{i\hat{K}}$의 해당 오비탈 회전에 큰 각도가 할당됩니다.

**Spin-balanced** 는 spin-α (spin-up)와 spin-β (spin-down) 섹터가 동일한 오비탈 회전 매개변수를 공유함을 의미합니다 — N₂처럼 closed-shell singlet (동일한 α와 β 점유)에 적합하며, 자유 매개변수 수를 절반으로 줄입니다.

### Interaction pairs — 하드웨어 제약

Jastrow factor는 두 큐비트가 직접 상호작용해야 합니다.
`interaction_pairs` 인수는 `ffsim`에 어떤 오비탈 쌍이 상호작용할 수 있는지 알려줍니다:

- `alpha_alpha_indices = [(p, p+1) ...]` — 큐비트 체인을 따라 동일 spin의 *최근접 이웃* 상호작용 (칩에서 spin 섹터당 한 줄씩).
- `alpha_beta_indices` — **교차 spin** 상호작용: 위치 $p$의 α 큐비트가 위치 $q$의 β 큐비트와 상호작용합니다.
  이는 **디바이스의 구조에 의해 제약**됩니다: 두 물리 큐비트가 하드웨어 상에서 서로 이웃하거나, 최대한 가까워야 합니다.

이것이 Exercise 1의 핵심입니다: IBM 디바이스마다 topology가 다르기 때문에, Heron 칩과 Nighthawk 칩에서 허용되는 `alpha_beta_indices`가 다릅니다.

In [ ]:
service = QiskitRuntimeService()
backend_hr = service.least_busy(simulator=False, operational=True, min_num_qubits=156)
plot_error_map(backend_hr)

### Jordan–Wigner 인코딩: 오비탈에서 큐비트로

**Jordan–Wigner (JW) 변환** 은 페르미온 오비탈 상태와 큐비트 상태를 연결하는 변환 규칙입니다:

$$\text{오비탈 } p \text{ 점유됨} \;\Longleftrightarrow\; \text{큐비트 } p = |1\rangle, \qquad
  \text{오비탈 } p \text{ 비어있음} \;\Longleftrightarrow\; \text{큐비트 } p = |0\rangle$$

총 $2 \times \texttt{num\_orbitals}$개의 큐비트를 사용합니다:
- 큐비트 $0, \ldots, \texttt{num\_orbitals}-1$ → spin-α 점유
- 큐비트 $\texttt{num\_orbitals}, \ldots, 2\cdot\texttt{num\_orbitals}-1$ → spin-β 점유

이것이 바로 Exercise 2a에서 `2*num_orbitals`의 bitstring이 α 절반과 β 절반으로 정확히 나뉘는 이유입니다.

두 가지 `ffsim` 회로 명령이 인코딩을 처리합니다:
- `PrepareHartreeFockJW` — 큐비트 초기 상태를 Hartree-Fock 점유 패턴으로 설정합니다 (처음 `num_elec_a`개의 α-큐비트와 `num_elec_b`개의 β-큐비트가 $|1\rangle$이고 나머지는 $|0\rangle$).
- `UCJOpSpinBalancedJW` — JW 큐비트 기저에서 UCJ 상관기를 적용합니다.

### PRE_INIT 패스와 게이트 수

`ffsim.qiskit.PRE_INIT` transpiler 패스는 메인 transpilation 패스가 실행되기 *전에* 고수준의 `ffsim` 게이트 객체를 그대로 최적화합니다.
`count_ops()`를 실행하면 PRE_INIT 유무에 따라 회로에 필요한 2-큐비트 게이트 수를 확인할 수 있습니다 — 게이트가 적을수록 노이즈가 줄어듭니다.

아래 코드는 **Heron** 디바이스에 대한 완전한 설정을 예시로 보여줍니다.
Exercise 1에서는 **Nighthawk (`ibm_miami`)** 디바이스에 대해 동일한 매핑을 수행합니다.

### SQD 워크플로우: 간략한 개요

회로가 준비되면, 나머지는 네 단계 과정을 따릅니다:

1. **샘플링** — 하드웨어에서 LUCJ 회로를 실행하고 bitstring을 수집합니다.
2. **구성 복원** — 노이즈로 인해 전자 수가 어긋난 bitstring을 수정합니다.
3. **부분 공간 구성** — 얻어진 bitstring을 다시 샘플링해 기저를 구성합니다.
4. **그 부분 공간에서 $\hat{H}$ 대각화** — 에너지 측정값을 얻고, 결과 오비탈 점유수를 2단계로 피드백합니다.

반복할수록 부분 공간이 점점 더 정교해집니다.

In [ ]:
# Heron 디바이스에서는 alpha-beta 상호작용이 네 인덱스마다 한 번씩 발생합니다
alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals - 1)]
alpha_beta_indices_hr = [(p, p) for p in range(0, num_orbitals, 4)]
alpha_beta_indices_hr = alpha_beta_indices_hr[:5]  # 다섯 번째 쌍에서 잘라냅니다

print(alpha_beta_indices_hr)

In [ ]:
n_reps = 1
ucj_op_hr = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t1=t1,
    t2=t2,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices_hr),
    optimize=True,
    options=dict(maxiter=20),
)
nelec = (num_elec_a, num_elec_b)

# 빈 양자 회로를 생성합니다
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit_hr = QuantumCircuit(qubits)

# Hartree-Fock 상태를 참조 상태로 준비하고 양자 회로에 추가합니다
circuit_hr.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# 참조 상태에 UCJ 연산자를 적용합니다
circuit_hr.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op_hr), qubits)

# 실행을 위해 회로를 살펴봅니다
inner_circuit_hr = ffsim.qiskit.PRE_INIT.run(circuit_hr)
for i in range(2, 0, -1):
    inner_circuit_hr.data.insert(i, CircuitInstruction(Barrier(num_qubits=2*num_orbitals), qubits))
display(inner_circuit_hr.decompose().draw("mpl", fold=-1))

# 최종 측정을 삽입합니다
circuit_hr.measure_all()

In [ ]:
# 디바이스 topology에 따라 spin a와 b의 layout을 선택합니다
spin_a_layout = [
     21,  36,  41,  42,  43,     56,  63,  64,  65,  77,
     85,  86,  87,  97, 107,    108, 109, 118, 129, 128,
    127, 126, 125, 117, 105,    104
]
spin_b_layout = [
     23,  24,  25,  37,  45,     46,  47,  57,  67,  68,
     69,  78,  89,  90,  91,     98, 111, 112, 113, 114,
    115,  99,  95,  94,  93,     79
]

initial_layout = spin_a_layout + spin_b_layout

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend_hr, initial_layout=initial_layout
)

# PRE_INIT 패스 없이
isa_circuit_hr = pass_manager.run(circuit_hr)
print(f"Gate counts (w/o pre-init passes): {isa_circuit_hr.count_ops()}")

# PRE_INIT 패스 적용
# 하드웨어 실행에는 이 pass manager로 생성된 회로를 사용합니다
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit_hr = pass_manager.run(circuit_hr)
print(f"Gate counts (w/ pre-init passes): {isa_circuit_hr.count_ops()}")

## Exercise 1: Nighthawk 디바이스에 LUCJ 회로 매핑하기

이제 동일한 회로를 **Nighthawk (`ibm_miami`)** 디바이스에 매핑해봅시다.
아래 error-map 셀을 실행하여 connectivity 다이어그램을 Heron과 비교해보세요.
α와 β 큐비트 레일이 어떻게 다르게 연결되어 있는지 확인하세요 — 이로 인해 하드웨어 네이티브 cross-spin interaction pair가 달라집니다.

> **`ibm_miami`에 접근할 수 없는 경우?** 대신 `FakeMiami`를 사용하세요 — 아래 셀에서 두 번째 줄의 주석을 해제하면 됩니다. `FakeMiami`는 Nighthawk 디바이스 구조 (동일한 큐비트 그래프와 basis 게이트)를 재현하므로 회로 매핑과 레이아웃 연습은 동일하게 작동합니다. 유일한 차이는 실제 하드웨어를 사용하는 대신 고전적인 방법으로 양자컴퓨터를 시뮬레이션한다는 점입니다.

In [ ]:
backend_nh = service.backend("ibm_miami")
# # ibm_miami에 접근할 수 없는 경우 FakeMiami를 사용하려면 이 명령을 사용하세요
# backend_nh = FakeMiami()

plot_error_map(backend_nh)

<a id="exercise_1a"></a>
<div class="alert alert-block alert-success">

<b>Exercise 1a: Nighthawk 디바이스를 위한 alpha–beta interaction pair 선택</b>

**목표:** Nighthawk 디바이스 구조에 맞는 cross-spin interaction pair를 `alpha_beta_indices_nh`에 채우세요.

**배경:** Section 2에서 `alpha_beta_indices`의 각 `(p, q)` 튜플은 "Jastrow factor가 α-오비탈 $p$와 β-오비탈 $q$를 결합하며, 두 큐비트가 칩에서 물리적으로 가까워야 한다"는 의미임을 떠올리세요.

- **Heron (`ibm_kingston`)** 에서는 α와 β 큐비트 체인이 4번째 사이트마다 만나기 때문에, `[(p, p) for p in range(0, num_orbitals, 4)]`를 5쌍으로 잘라 사용했습니다 (하드웨어 인접 α–β 교차점이 5개만 있습니다).
- **Nighthawk (`ibm_miami`)** 에서는 α와 β 레일을 나란히 배치할 수 있어, α 체인의 $p$번째 큐비트가 β 체인의 $p$번째 큐비트와 *모든* $p$에 대해 상호작용할 수 있습니다. 인덱스를 건너뛸 필요가 없습니다.

**힌트:**
- **(권장)** 모든 큐비트 쌍을 연결하는 대신 24개의 큐비트 쌍만을 만드세요 (24에서 잘라내기). 이렇게 하면 Nighthawk 디바이스에 회로를 더 쉽게 매핑할 수 있습니다.
- 모든 α와 β 쌍이 직접 인접하지는 못할 수도 있습니다만, 이 단계에서는 걱정하지 않아도 됩니다.

</div>

In [ ]:
# Nighthawk 디바이스 구조에 따라 alpha-beta interaction pair를 선택합니다
# ---- TODO : Task 1a ----
alpha_beta_indices_nh = []
# ---- End of TODO : Task 1a ----

print(alpha_beta_indices_nh)

In [ ]:
grade_lab4c_ex1a(alpha_beta_indices_nh)

In [ ]:
ucj_op_nh = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t1=t1,
    t2=t2,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices_nh),
    optimize=True,
    options=dict(maxiter=20),
)
nelec = (num_elec_a, num_elec_b)

# 빈 양자 회로를 생성합니다
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit_nh = QuantumCircuit(qubits)

# Hartree-Fock 상태를 참조 상태로 준비하고 양자 회로에 추가합니다
circuit_nh.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# 참조 상태에 UCJ 연산자를 적용합니다
circuit_nh.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op_nh), qubits)

# 실행을 위해 회로를 살펴봅니다
inner_circuit_nh = ffsim.qiskit.PRE_INIT.run(circuit_nh)
for i in range(2, 0, -1):
    inner_circuit_nh.data.insert(i, CircuitInstruction(Barrier(num_qubits=2*num_orbitals), qubits))
display(inner_circuit_nh.decompose().draw("mpl", fold=-1))

# 최종 측정을 삽입합니다
circuit_nh.measure_all()

<a id="exercise_1b"></a>
<div class="alert alert-block alert-success">

<b>Exercise 1b: Nighthawk 디바이스를 위한 큐비트 레이아웃 정의</b>

**목표:** α와 β 스핀 오비탈 체인을 Nighthawk 하드웨어에 올바르게 매핑하는 `spin_a_layout`과 `spin_b_layout` — 각각 정확히 `num_orbitals` (= 26)개의 물리 큐비트 인덱스 리스트 — 를 제공하세요.

**배경:** `initial_layout`은 transpiler에게 논리 큐비트 $p$를 물리 큐비트 `initial_layout[p]`에 배치하도록 지시합니다. 좋은 레이아웃은 두 가지 요건을 동시에 만족해야 합니다:

1. **각 spin 섹터 내 connectivity:** 동일 spin 최근접 이웃 상호작용 `alpha_alpha_indices = [(p, p+1) ...]`은 각 리스트 내에서 물리 큐비트 $p$와 $p+1$이 하드웨어 상에서 인접(칩의 2-큐비트 게이트 선으로 연결)해야 합니다. 따라서 각 리스트는 디바이스 그래프를 통과하는 *연결된 경로*를 형성해야 합니다.

2. **α–β 인접성:** Exercise 1a에서 제공한 쌍에 따라, 가능한 한 쌍을 서로 가깝게 배치하세요. 필요하다면 몇 개의 swap을 사용해도 됩니다.

**힌트:**
- 모든 α와 β 쌍이 인접하여 배치되지는 않을 수 있습니다. swap 연산을 몇 개 적용하는 것은 괜찮으며, swap들이 병렬로 실행될 수 있다면 회로 깊이를 최소화할 수 있습니다.
- **(선택 사항)** 더 고급 도전을 원한다면, 이전 연습에서 26개의 큐비트를 모두 쌍으로 연결하고 Nighthawk에 매핑해보세요. 통과 기준이 상당히 까다롭고 transpiler seed에 따라 grader 결과가 달라질 수 있습니다. 더 나은 transpilation 결과를 위해 몇 가지 다른 seed를 시도해보세요.
</div>

In [ ]:
# 디바이스 topology에 따라 spin a와 b의 layout을 선택합니다
# ---- TODO : Task 1b ----
spin_a_layout = [

]
spin_b_layout = [
    
]
initial_layout = spin_a_layout + spin_b_layout
# ---- End of TODO : Task 1b ----

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend_nh, initial_layout=initial_layout
)

# PRE_INIT 패스 없이
isa_circuit_nh = pass_manager.run(circuit_nh)
print(f"Gate counts (w/o pre-init passes): {isa_circuit_nh.count_ops()}")

# PRE_INIT 패스 적용
# 하드웨어 실행에는 이 pass manager로 생성된 회로를 사용합니다
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit_nh = pass_manager.run(circuit_nh)
print(f"Gate counts (w/ pre-init passes): {isa_circuit_nh.count_ops()}")

In [ ]:
grade_lab4c_ex1b(initial_layout, alpha_beta_indices_nh, seed=None)

## 3. 양자 샘플링

회로를 Nighthawk 토폴로지에 매핑했으면, 이제 디바이스에 2,000 shot을 실행합니다.
각 **shot** 은 `2*num_orbitals`개의 큐비트를 모두 측정하여 하나의 **bitstring** — 어떤 오비탈이 점유되어 있는지를 나타내는 이진 스냅샷 — 을 반환합니다.
물리적으로, 각 bitstring은 *하나의 제안된 전자 배치*입니다: 기저 상태의 후보 basis state입니다.

이 2,000개의 bitstring 전체와 각 배치의 등장 빈도가 SQD의 입력 데이터입니다.

> **실행 시간 참고:** 실제 하드웨어 작업은 `ibm_miami`에서 10초, `ibm_kingston`에서 2초가 소요됩니다.
> 이 노트북을 독립적으로 사용할 수 있도록, 아래 셀은 작업을 제출하고 `job_id`를 저장합니다.
> *다음* 셀은 저장된 ID에서 미리 계산된 결과를 가져옵니다 — 재제출할 필요가 없습니다.

In [ ]:
# # 옵션 1: Nighthawk 디바이스에 접근 권한이 있다면, square-lattice 회로를 실행해보세요.
# sampler = Sampler(mode=backend_nh)
# job = sampler.run([isa_circuit_nh], shots=2_000)

# # 옵션 2: Nighthawk 디바이스에 접근 권한이 없다면, Heron 회로를 실행할 수 있습니다.
# sampler = Sampler(mode=backend_hr)
# job = sampler.run([isa_circuit_hr], shots=2_000)

job_id = job.job_id()
print(f"Submitted job {job_id}")

아래 셀은 완료된 작업을 가져와 측정 결과를 두 배열로 변환합니다:

- `raw_bitstrings` — 각 행이 고유한 bitstring인 형태 `(n_unique, 2*num_orbitals)`의 정수 배열 (각 원소는 0 또는 1).
- `raw_probs` — 각 bitstring의 경험적 확률을 담은 길이 `n_unique`의 float 배열. `raw_probs[k]`는 `raw_bitstrings[k]`를 생성한 shot의 비율입니다.

In [ ]:
# job id를 사용하여 작업 가져오기
job_id = "PutYourJobIdPrintedAbove"
job = service.job(job_id)
primitive_result = job.result()
pub_result = primitive_result[0]
bit_array = pub_result.data.meas

# BitArray를 bitstring 및 확률 배열로 변환
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

## Exercise 2: 결과 bitstring 살펴보기

샘플링이 끝났으면, 이제 결과를 살펴볼 차례입니다.
아래 두 코드 셀로 처리를 시작하기 전에 `raw_bitstrings`의 형태와 내용을 확인합니다.
먼저 실행한 후, 두 가지 연습 문제를 풀어보세요.

In [ ]:
print(raw_bitstrings.shape)

In [ ]:
print(raw_bitstrings[0])

<a id="exercise_2a"></a>
<div class="alert alert-block alert-success">

<b>Exercise 2a: Raw bitstring을 스핀 섹터로 재구성하기</b>

**목표:** 실험 결과에서 **spin-α** 와 **spin-β** bitstring을 분리하는 `reshape_bitstring` 함수를 구현하세요.

**배경:** Jordan-Wigner 인코딩은 각 bitstring의 *첫 번째* 블록에 spin-α 큐비트를, *두 번째* 블록에 spin-β 큐비트를 배치합니다.
α 블록의 위치 $p$에 `1`이 있으면 "α-오비탈 $p$가 점유됨"을, `0`은 비어 있음을 의미합니다.  
두 스핀 섹터를 분리하면:
- 각 스핀별로 전자 수를 독립적으로 확인할 수 있습니다 (Exercise 2b).
- Configuration recovery 중에 스핀별 보정을 적용할 수 있습니다 (Exercise 3).

**힌트:**
- `raw_bitstrings`의 형태는 **(2000, 52)** 입니다. **(2000, 2, 26)** 형태의 배열을 얻어야 합니다.
- **비트 순서 주의:** Qiskit은 측정 결과를 *little-endian* 형식으로 반환합니다 (마지막으로 측정된 큐비트 = 문자열의 첫 번째 비트). 재구성 후 `reshaped_bitstrings[0, :]`를 출력하여 어떤 블록이 α인지 확인하세요. 스핀 블록 순서를 뒤집어 인덱스 0이 항상 α가 되도록 해야 할 수 있습니다.
- 위에서 출력된 `raw_bitstrings[0]`과 교차 확인하세요.

</div>

In [ ]:
def reshape_bitstring(
    bitstrings: np.ndarray,
    num_orbitals: int
) -> np.ndarray:

    # ---- TODO : Task 2a ----

    # ---- End of TODO : Task 2a ----

    return reshaped_bitstrings

reshaped_bitstrings = reshape_bitstring(raw_bitstrings, num_orbitals)
print(reshaped_bitstrings[0, :])

In [ ]:
grade_lab4c_ex2a(reshape_bitstring, raw_bitstrings)

<a id="exercise_2b"></a>
<div class="alert alert-block alert-success">

<b>Exercise 2b: 샘플별 전자 수 세기 (Hamming weight)</b>

**목표:** 재구성된 bitstring 배열을 받아, 각 샘플의 각 spin sector에서 점유된 오비탈 수를 나타내는 정수 배열을 반환하는 `hamming_weight` 함수를 구현하세요.

**배경:** 이진 문자열의 **Hamming weight** 는 단순히 1의 개수입니다.
이 맥락에서 Hamming weight = 해당 샘플에서 해당 spin의 전자 수입니다.

*유효한* 샘플은 정확히 5개의 spin-α 전자와 5개의 spin-β 전자를 가져야 합니다 (입자 수 보존).
노이즈 없는 환경에서는 모든 shot이 유효합니다. 실제 하드웨어에서는 비트 플립 오류와 기타 노이즈 메커니즘으로 인해 일부 bitstring이 올바른 particle-number sector를 벗어납니다 — 이런 샘플은 물리적으로 무의미하며, 그대로 대각화에 사용되면 결과를 오염시킬 수 있습니다.

Hamming weight 계산은 이러한 불량 샘플을 감지하는 첫 번째 단계입니다.

**힌트:** 마지막 축 (오비탈 축) 방향으로 합산해보세요.

</div>

In [ ]:
def hamming_weight(bitstrings: np.ndarray) -> np.ndarray:

    # ---- TODO : Task 2b ----

    # ---- End of TODO : Task 2b ----

    return weight

print(hamming_weight(reshaped_bitstrings)[0])

In [ ]:
grade_lab4c_ex2b(hamming_weight)

아래 셀은 전자 수가 올바른 샘플, 즉 보정 전 "유효한" shot이 얼마나 되는지 확인합니다. 이 수가 0이어도 놀라지 마세요 — 실제 하드웨어의 비트 플립 오류로 인해 샘플이 올바른 particle-number sector를 쉽게 벗어날 수 있습니다.

In [ ]:
print(int(2e3*np.sum(raw_probs[np.all(hamming_weight(reshaped_bitstrings) == nelec)])))

## Exercise 3: bitstring 복원하기

### Configuration recovery가 필요한 이유

분자의 진짜 기저 상태는 완전히 **고정된 particle-number sector** 안에 존재합니다: 정확히 `(num_elec_a, num_elec_b)`개의 전자를 가진 배치들입니다.
양자역학은 입자 수를 보존합니다 — 물리적인 분자는 자발적으로 전자를 얻거나 잃을 수 없습니다.

하지만 실제 양자 디바이스는 일부 `0`을 `1`로, `1`을 `0`으로 바꾸는 **비트 플립 오류** (및 기타 노이즈) 를 발생시킵니다.
비트 플립이 단 하나만 일어나도 bitstring은 물리적 sector를 벗어나고, 해당 샘플은 잘못된 전자 수를 보고하게 됩니다.
그 샘플은 물리적으로 무의미하며, 대각화기에 직접 입력하면 비물리적 배치로 부분공간을 오염시킵니다.

**Configuration recovery** 는 각 불량 샘플을 수정할 때 올바른 전자 수를 복원하는 데 필요한 최소한의 비트만 뒤집습니다. 어떤 비트를 뒤집을지는 각 오비탈의 평균 occupancy에 대한 현재 최선의 측정값을 기반으로 선택합니다.

### Hartree-Fock occupancy로 초기화하기

Configuration recovery 알고리즘에는 "각 오비탈이 평균적으로 얼마나 점유되어 있는가?"에 대한 사전 측정값이 필요합니다.
이를 **Hartree-Fock occupancy** 로 초기화합니다: `num_elec_a`개의 가장 높은 인덱스 오비탈은 완전히 점유(occupancy = 1)되고, 나머지는 비어 있습니다(occupancy = 0).
SQD 루프가 진행되면서 이 값은 대각화된 양자 상태에서 측정된 실제 오비탈 occupancy로 갱신됩니다 — 따라서 recovery는 반복할수록 더 정확해집니다.

In [ ]:
initial_occupancy = np.zeros((2, num_orbitals), dtype=float)
initial_occupancy[:, -num_elec_a:] = 1.
print(initial_occupancy)

<a id="exercise_3a"></a>
<div class="alert alert-block alert-success">

<b>Exercise 3a: 비트 플립 확률 함수 정의하기</b>

**목표:** $[0, 1]$ 범위의 오비탈 occupancy 배열을 받아, 각 오비탈을 비어 있음(0)에서 점유됨(1)으로 뒤집을 확률 가중치를 반환하는 `weight_flip_0_to_1(occupancy)` 함수를 구현하세요.

**배경:** 샘플에 전자가 *부족할* 때(1이 너무 적을 때), 빈 오비탈 일부를 점유 상태로 뒤집어야 합니다.
이때 *점유되어 있을 확률이 높은* 오비탈을 우선적으로 채우는 것이 바람직합니다 — 진짜 기저 상태에서는 그 오비탈들이 채워져 있을 가능성이 높기 때문입니다.
반대로, 보통 비어 있는 오비탈은 거의 채우지 않아야 합니다.

함수:

$$W_{0\to1}(\text{occ}) = e^{\text{occ}} - 1, \qquad 0 \le \text{occ} \le 1$$

는 이 동작을 정확히 포착합니다:
- $W_{0\to1}(0) = 0$ — 한 번도 점유된 적 없는 오비탈은 절대 채우지 않습니다.
- $W_{0\to1}(1) = e - 1 \approx 1.718$ — 항상 점유된 오비탈을 강하게 선호합니다.
- 그 사이에서 단조 증가합니다.

짝 함수 `weight_flip_1_to_0(occ) = weight_flip_0_to_1(1 - occ)`는 이미 제공되어 있습니다. 전자를 제거할 때는 비어 있을 확률이 높은 (0일 확률이 높은) 오비탈을 선호합니다.

</div>

In [ ]:
def weight_flip_0_to_1(occupancy: np.ndarray) -> np.ndarray:

    # ---- TODO : Task 3a ----

    # ---- End of TODO : Task 3a ----

    return prob

def weight_flip_1_to_0(occupancy):
    return weight_flip_0_to_1(1-occupancy)

In [ ]:
grade_lab4c_ex3a(weight_flip_0_to_1)

<a id="exercise_3b"></a>
<div class="alert alert-block alert-success">

<b>Exercise 3b: bitstring configuration을 올바른 전자 수로 복원하기</b>

**목표:** per-spin 루프 내부의 세 가지 분기 보정 로직을 채워 `recover_configurations` 함수를 완성하세요.

**배경 — 알고리즘, 단계별:**

외부 루프는 모든 bitstring 샘플을 순회합니다. 각 샘플의 각 스핀 섹터 `i`(0 = α, 1 = β)에 대해:

1. `weight[i]` = Hamming weight = sector `i`의 전자 수를 계산합니다(Exercise 2b에서 작성한 함수 사용).
2. `weight[i]`를 `num_elec[i]`와 비교합니다:

   - **전자가 너무 적을 때** (`weight[i] < num_elec[i]`): 빈 비트 `num_elec[i] - weight[i]`개를 1로 뒤집어야 합니다.
     - 빈 오비탈 찾기
     - 해당 후보들의 flip 가중치 구하기
     - 가중치를 정규화하여 확률 분포 만들기
     - 뒤집을 비트를 무작위로 선택
     - 선택된 비트를 `True`로 설정

   - **전자가 너무 많을 때** (`weight[i] > num_elec[i]`): 점유된 비트 `weight[i] - num_elec[i]`개를 0으로 뒤집어야 합니다.
     - 위 절차를 반대로 적용하되, 뒤집을 점유된 비트를 선택합니다.

   - **정확히 맞을 때** (`weight[i] == num_elec[i]`): 스핀 섹터를 변경하지 않습니다.

**힌트:** 각 분기는 네 줄로 구성됩니다: `np.where`로 후보 찾기, 확률 계산 및 정규화, `rand_seed.choice`로 뒤집을 비트 선택, 비트 뒤집기.

**참고:** 출력 형식: 보정된 각 bitstring은 문자열로 변환되어 `corrected_dict`(`defaultdict`)에 누적됩니다 — 보정 후 동일해진 bitstring은 병합됩니다.

</div>

In [ ]:
def recover_configurations(
    bitstrings: np.ndarray,
    probs: np.ndarray,
    occupancy: np.ndarray,
    num_orbitals: int,
    num_elec: tuple[int, int],
    rand_seed: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    corrected_dict: defaultdict[str, float] = defaultdict(float)
    for bitstring, freq, weight in zip(bitstrings, probs, hamming_weight(bitstrings)):
        bs_corrected = bitstring.copy()
        for i in range(2):
            # ---- TODO : Task 3b ----



            # ---- End of TODO : Task 3b ----
        bs_str = "".join("1" if bit else "0" for bit in bs_corrected.flatten())
        corrected_dict[bs_str] += freq

    bs_mat_out = np.array([[bit == "1" for bit in bs] for bs in corrected_dict]).reshape(-1, 2, num_orbitals)
    freqs_out = np.array([f for f in corrected_dict.values()])
    freqs_out = np.abs(freqs_out) / np.sum(np.abs(freqs_out))

    return bs_mat_out, freqs_out

recovered_bitstrings, recovered_probs = recover_configurations(
    reshaped_bitstrings, raw_probs, initial_occupancy, num_orbitals, nelec, np.random.default_rng())

아래 셀에서 configuration recovery **이후** 유효한 샘플 수(올바른 전자 수를 가진 샘플)를 확인할 수 있습니다 — Exercise 2b의 보정 전 수치와 비교해 보세요.

In [ ]:
print(np.rint(2e3*np.sum(recovered_probs[np.all(hamming_weight(recovered_bitstrings) == nelec, axis=1)])).astype(int))

In [ ]:
grade_lab4c_ex3b(recover_configurations)

### 복원된 bitstring에서 대각화 부분공간으로

모든 샘플이 올바른 particle-number sector에 속하면, 대각화를 위한 준비를 합니다:

1. **`subsample`** 은 복원된 구성들을 `samples_per_batch`개의 샘플로 이루어진 `num_batches`개의 독립 배치로 분할합니다. 여러 독립 배치에서 eigensolver를 실행하면 분산 측정이 가능하고(나중에 박스 그래프가 그 분포를 보여줍니다), 운이 없는 구성 집합에 갇힐 가능성도 줄어듭니다.

2. **`bitstring_matrix_to_integers`** 는 각 이진 bitstring을 단일 Python 정수로 압축합니다. 이는 selected-CI eigensolver인 `solve_sci_batch`가 요구하는 표준 "determinant label" 형식입니다.
   이 정수 레이블은 **CI strings**(Configuration Interaction strings)라고 합니다: 각 정수는 어떤 오비탈이 점유되어 있는지를 인코딩하여 하나의 **Slater determinant** 를 고유하게 식별합니다. Slater determinant란 파울리 배타 원리를 만족하는 유효한 electron configuration 하나를 나타내는, 단일 전자 오비탈 상태들의 올바르게 반대칭화된 곱입니다.

3. **`build_subspace`** 는 CI strings를 입력받아 각 배치 내에서 중복을 제거하고, 주변 확률(가장 많이 샘플링된 순서)로 정렬하며, 선택적으로 사용자가 지정한 `include` 집합을 앞에 추가한 뒤, spin당 `max_dim`개의 구성으로 잘라냅니다. 결과는 **부분공간의 기저**입니다: 해밀토니안이 투영될 Slater determinant들의 집합입니다.

In [ ]:
subsamples = np.array(subsample(recovered_bitstrings.reshape(-1, 2*num_orbitals), recovered_probs, samples_per_batch=20, num_batches=5))
subsamples = subsamples.reshape(*subsamples.shape[:-1], 2, -1)
print(subsamples.shape)
print(subsamples[4, 19])

In [ ]:
ci_strs = np.array([bitstring_matrix_to_integers(samples[:, i]) for samples in subsamples for i in range(2)])
ci_strs = ci_strs.reshape(-1, 2, ci_strs.shape[-1])
print(ci_strs.shape)
print(ci_strs[0])

In [ ]:
def build_subspace(subsamples, include=np.empty((2, 0), dtype=int), max_dim=(None, None)):

    subspaces = []
    for samples in subsamples:
        subspace_list = []
        for i in range(2):

            # 단일 spin bitstring과 개수를 가져옵니다.
            bitstrings, counts = np.unique(samples[i], return_counts=True)

            # 단일 spin bitstring을 주변 확률 내림차순으로 정렬합니다.
            bitstrings = bitstrings[np.argsort(counts)[::-1]]
            
            # 명시적으로 요청된 bitstring을 우선시하고, 그다음 샘플링된 bitstring을 붙입니다.
            subspace = np.concatenate((include[i], bitstrings))

            # 원래 순서를 유지하면서 고유한 값을 가져옵니다.
            _, indices = np.unique(subspace, return_index=True)
            indices.sort()
            subspace = subspace[indices]

            # bitstring을 최대 차원으로 잘라냅니다.
            subspace = subspace[:max_dim[i]]
            subspace.sort()

            subspace_list.append(subspace)
        subspaces.append(subspace_list)

    return subspaces

subspace = build_subspace(ci_strs, max_dim=(15, 15))
print(np.array(subspace).shape)

### SQD 루프 파라미터

SQD 루프를 실행하기 전에, 각 파라미터가 무엇을 제어하는지 설명합니다:

| 파라미터 | 의미 |
|---|---|
| `max_iterations` | 복원 → 서브샘플링 → 대각화 사이클을 몇 번 실행할지. 반복이 많을수록 occupancy 측정이 수렴합니다. |
| `num_batches` | 반복당 독립 부분공간 배치 수. 배치가 많을수록 통계가 풍부해집니다. |
| `samples_per_batch` | 배치당 구성 수. 클수록 부분공간이 풍부해지지만 고전 eigensolver 작업량이 늘어납니다. |
| `max_dim` | 부분공간 내 spin sector당 Slater determinant의 최대 수. 해밀토니안 행렬의 차원을 제어합니다. |

In [ ]:
# 시드
seed=142

# SQD 옵션
max_iterations = 5

# Eigenstate solver 옵션
num_batches = 5
samples_per_batch = 500
max_dim = (300, 300)
max_cycle = 200

# 내장 eigensolver에 옵션을 전달합니다. 기본값을 사용하려면
# 이 단계를 생략하면 되며, 그 경우 아래 diagonalize_fermionic_hamiltonian 호출에서
# sci_solver 인수를 지정하지 않으면 됩니다.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# 중간 결과를 저장할 리스트
result_history = []

def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i+1}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}")

### 자기 일관 SQD 루프

아래 루프는 4단계 SQD 사이클을 처음부터 끝까지 구현합니다:

1. **복원(Recover)** — 현재 `current_occupancies`를 flip 확률 사전 분포로 사용하여 `recover_configurations`를 호출합니다.
2. **서브샘플링(Subsample)** — `samples_per_batch`개의 구성으로 이루어진 `num_batches`개의 독립 배치를 추출합니다.
3. **부분공간 구성** — CI strings로 변환하고 `build_subspace`를 호출하여 Slater determinant basis를 얻습니다.
4. **대각화(Diagonalize)** — `sci_solver`를 호출하여 해당 부분공간에서 $\hat{H}$를 투영하고 대각화합니다.

대각화 이후의 핵심 피드백 단계는 다음과 같습니다:
```python
current_occupancies = current_result.orbital_occupancies
```
solver가 생성한 고유벡터는 근사 바닥 상태에서 각 오비탈이 평균적으로 얼마나 점유되는지 알려줍니다.
이 갱신된 occupancy 사전 분포는 *다음* 반복의 1단계로 피드백되어 구성 복원이 점점 더 정확해집니다.
루프는 `max_iterations` 사이클 동안 실행되며, 모든 배치와 반복에 걸쳐 에너지가 가장 낮은 결과를 추적합니다.

In [ ]:
result_history = []

rng = np.random.default_rng(seed)
current_occupancies = initial_occupancy
best_result = None
sci_solver = solve_sci_batch

# BitArray를 bitstring 및 확률 배열로 변환합니다.
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

# 구성 복원 루프를 실행합니다.
for _ in range(max_iterations):
    reshaped_bitstrings = reshape_bitstring(raw_bitstrings, num_orbitals)

    # 평균 오비탈 occupancy 정보가 있는 경우, 이를 사용하여
    # 노이즈가 있는 전체 구성 집합을 정제합니다.
    bitstrings, probs = recover_configurations(
        reshaped_bitstrings, raw_probs, current_occupancies, num_orbitals, nelec, rand_seed=rng
    )

    # bitstring 배치를 서브샘플링합니다.
    subsamples = np.array(subsample(
        bitstrings.reshape(-1, 2*num_orbitals),
        probs,
        samples_per_batch=samples_per_batch,
        num_batches=num_batches,
        rand_seed=rng,
    ))
    subsamples = subsamples.reshape(*subsamples.shape[:-1], 2, -1)

    ci_strs = np.array([bitstring_matrix_to_integers(samples[:, i]) for samples in subsamples for i in range(2)])
    ci_strs = ci_strs.reshape(-1, 2, ci_strs.shape[-1])

    # bitstring을 CI strings로 변환하고 최대 차원에 따라 잘라냅니다.
    subspace = build_subspace(ci_strs, max_dim=max_dim)

    # 대각화를 실행합니다.
    results = sci_solver(subspace, hcore, eri, num_orbitals, nelec)

    # 콜백 함수를 호출합니다.
    callback(results)

    # 배치에서 가장 좋은 결과를 가져옵니다.
    best_result_in_batch = min(results, key=lambda result: result.energy)

    # 지금까지 본 에너지 중 가장 낮은지 확인합니다.
    if best_result is None or best_result_in_batch.energy < best_result.energy:
        best_result = copy.deepcopy(best_result_in_batch)

    # 현재 결과와 occupancy를 저장합니다.
    current_result = copy.deepcopy(best_result_in_batch)
    current_occupancies = current_result.orbital_occupancies

# 최소 한 번의 반복이 있었으므로 best_result는 None이 아닙니다.
best_result = cast(SCIResult, best_result)

In [ ]:
print(f"Best energy = {best_result.energy + nuclear_repulsion_energy}")

plt.axhline(scf.e_tot, color="C0", label="HF")
plt.axhline(ccsd.e_tot, color="C1", label="CCSD")
plt.boxplot(
    [x.energy + nuclear_repulsion_energy for results in result_history for x in results],
    tick_labels=['Original SQD'],
)
plt.legend(loc='upper right')
plt.show()

# 가장 좋은 부분공간 basis를 제출합니다.
best_subspace = (best_result.sci_state.ci_strs_a, best_result.sci_state.ci_strs_b)

원래의 SQD 방법이 Hartree-Fock 상태보다 더 나은 에너지 측정값을 찾는 것을 확인할 수 있습니다. 이는 양자 샘플들이 대각화를 Hilbert space의 물리적으로 의미 있는 영역으로 유도하고 있음을 보여주는, 이미 자명하지 않은 결과입니다.
그러나 결과는 아직 고전적인 CCSD 벤치마크에는 미치지 못합니다.

SQD와 CCSD 사이의 격차에는 여러 원인이 있을 수 있습니다. 가장 직접적인 조절 수단은 **부분공간의 크기**입니다: 부분공간이 클수록 바닥 상태 파동함수를 더 많이 포착하지만, 대각화에 훨씬 더 많은 고전 계산이 필요합니다. 규모를 키우려면 고성능 워크스테이션부터 전체 HPC 클러스터까지도 필요할 수 있습니다.

여기서는 대신 이 격차의 또 다른 주요 원인인 순수 샘플링의 한계에 초점을 맞춥니다. 하드웨어 노이즈와 유한한 shot 수로 인해 샘플러는 화학적으로 중요한 일부 electron configuration을 놓칩니다. 특히 Hartree-Fock 기준 상태와 오비탈 occupancy가 단 하나 또는 두 개만 다른 구성들이 그렇습니다. 이러한 HF에 가까운 구성들은 바닥 상태에 강하게 기여하지만, 하드웨어 노이즈로 인해 측정 분포에서 억제될 수 있습니다.

**Exercise 4는 고전적으로 열거된 기준 determinant 집합으로 샘플링된 부분공간을 보강하여 이 격차를 직접 해결합니다.**

## Exercise 4: Augmented SQD with a classical reference subspace

### Reference subspace

고전적 부분공간과 양자 부분공간을 결합하기 전에, 먼저 양자 샘플 없이 *순수하게 고전적이고 물리적으로 동기 부여된* 부분공간에서 대각화하면 어떤 결과가 나오는지 살펴보겠습니다.

Background에서 소개한 것처럼, **classical selected-CI** 는 화학자들이 섭동 이론이나 물리적 직관을 사용하여 가장 영향력 있는 Slater determinant들을 반복적으로 식별하고 포함하는 방법들의 모음입니다. 결과의 정확도는 해당 선택의 품질에 달려 있습니다.

**이 랩의 reference subspace** 는 이 방법의 더 단순한 brute-force 근사입니다. 어떤 선택 기준도 적용하지 않고, 낮은 에너지부터 오비탈을 채우는 방식으로 낮은 excitation rank의 상태들을 모두 열거하여 처음 300개를 유지합니다. 이러한 **reference determinant** 들은 화학적 통찰 없이도 구성할 수 있으며, 체계적인 열거만으로 재현 가능하고 계산 비용이 저렴한 고전 베이스라인이 됩니다.

Background에서 언급했듯이, 이 brute-force 방법은 일반적으로 HF보다는 좋지만 CCSD에는 미치지 못하는 에너지를 달성합니다. 이것이 **양자 유용성(quantum utility)을 위한 고전 베이스라인** 역할을 합니다: SQD 결과가 이를 능가한다면, 양자 샘플들이 고전적인 열거법으로는 찾을 수 없는 상태들을 찾은 것입니다.

이 determinant들을 `itertools.combinations`를 사용하여 체계적으로 열거합니다:

In [ ]:
from itertools import combinations

print(list(combinations(range(5), 3)))

<a id="exercise_4"></a>
<div class="alert alert-block alert-success">

<b>Exercise 4: Build a classical reference subspace</b>

**목표:** `ref_bitstrings`를 구성하세요. 이는 Hartree-Fock 기준 주변의 가장 낮은 excitation rank 구성을 나타내는 최대 300개의 Slater determinant이며, eigensolver에 적합한 `ref_ci_strings`로 변환해야 합니다.

**배경:** 연주회장 비유로 돌아가서: 양자 샘플러가 우연히 가장 중요한 좌석 배치를 발견하기를 기대하는 대신, 전자들이 가장 낮은 오비탈부터 채우는 *앞줄* 구성들을 체계적으로 열거하여 eigensolver에 직접 전달합니다. 화학자들이 섭동 이론을 사용하여 가장 영향력 있는 determinant를 식별하는 진정한 classical selected-CI와 달리, 이는 brute-force 열거법입니다: 선택 기준 없이 가장 낮은 excitation rank의 구성 중 처음 300개만 사용합니다. Background에서 언급했듯이, 일반적으로 HF와 CCSD 사이의 에너지를 산출하여 넘어서야 할 고전 베이스라인이 됩니다.

**힌트:**
- 완전히 점유된 bitstring에서 시작하여 에너지가 가장 높은 오비탈부터 비워나가는 방식을 고려해 보세요.
- CI string 형식으로 변환하려면 `bitstring_matrix_to_integers`를 적용하세요.
- 대각화기에 spin-α와 spin-β strings를 모두 제공해야 한다는 점을 기억하세요.

</div>

In [ ]:
# ---- TODO : Task 4 ----



# ---- End of TODO : Task 4 ----

In [ ]:
grade_lab4c_ex4(ref_ci_strings)

이제 순수하게 고전적인 reference subspace에서 해밀토니안을 대각화해 보세요:


In [ ]:
ref_result = sci_solver([ref_ci_strings], hcore, eri, num_orbitals, nelec)[0]
print(f"Reference Subspace Energy: {ref_result.energy + nuclear_repulsion_energy}")
print(f"Subspace dimension: {np.prod(ref_result.sci_state.amplitudes.shape)}")

Reference subspace 에너지가 원래의(순수 양자 샘플링) SQD 결과보다 **낮을** 수 있습니다. 이는 선별된 저에너지 구성들이 바닥 상태에 진정으로 중요하며, 양자 샘플러만으로는 하드웨어 노이즈로 인해 그 중 일부를 놓칠 수 있음을 확인해 줍니다.

### 양자 샘플과 고전적 reference subspace의 결합

가장 강력한 접근법은 두 소스를 결합하는 것입니다:
- 기준 determinant 집합을 보장된 앵커로 **고정(Pin)** 합니다(`include = ref_ci_strings[:, :100]` — 처음 100개의 구성).
- 양자 샘플러가 나머지 `max_dim - 100`개의 구성을 배치마다 평소처럼 제공하도록 합니다.

이렇게 하면 모든 대각화가 탄탄한 고전적 기반에서 시작되고, 양자 장치는 공간의 보완적인 부분을 탐색합니다.
결과가 순수 샘플링 및 순수 reference subspace 에서의 결과와 비교하여 어떻게 달라질 것으로 예상하시나요?


In [ ]:
# 중간 결과를 저장할 리스트
pm_result_history = []

def callback(results: list[SCIResult]):
    pm_result_history.append(results)
    iteration = len(pm_result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}")

# SQD 루프 동안 100개의 가장 낮은 excitation rank reference determinant를 고정합니다.
# 나머지 항목은 평소처럼 양자 샘플에서 채워집니다.
include = ref_ci_strings[:, :100]  # 100개의 reference determinant를 고정합니다.

rng = np.random.default_rng(seed)
current_occupancies = initial_occupancy
best_result = None
sci_solver = solve_sci_batch

# BitArray를 bitstring 및 확률 배열로 변환합니다.
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

# 구성 복원 루프를 실행합니다.
for _ in range(max_iterations):
    reshaped_bitstrings = reshape_bitstring(raw_bitstrings, num_orbitals)

    # 평균 오비탈 occupancy 정보가 있는 경우, 이를 사용하여
    # 노이즈가 있는 전체 구성 집합을 정제합니다.
    bitstrings, probs = recover_configurations(
        reshaped_bitstrings, raw_probs, current_occupancies, num_orbitals, nelec, rand_seed=rng
    )

    # bitstring 배치를 서브샘플링합니다.
    subsamples = np.array(subsample(
        bitstrings.reshape(-1, 2*num_orbitals),
        probs,
        samples_per_batch=samples_per_batch,
        num_batches=num_batches,
        rand_seed=rng,
    ))
    subsamples = subsamples.reshape(*subsamples.shape[:-1], 2, -1)

    ci_strs = np.array([bitstring_matrix_to_integers(samples[:, i]) for samples in subsamples for i in range(2)])
    ci_strs = ci_strs.reshape(-1, 2, ci_strs.shape[-1])

    # bitstring을 CI strings로 변환하고 요청된 include strings를 포함하여
    # max_dim 으로 잘라냅니다.
    subspace = build_subspace(ci_strs, include, max_dim=max_dim)

    # 대각화를 실행합니다.
    results = sci_solver(subspace, hcore, eri, num_orbitals, nelec)

    # 콜백 함수를 호출합니다.
    callback(results)

    # 배치에서 가장 좋은 결과를 가져옵니다.
    best_result_in_batch = min(results, key=lambda result: result.energy)

    # 지금까지 본 에너지 중 가장 낮은지 확인합니다.
    if best_result is None or best_result_in_batch.energy < best_result.energy:
        best_result = copy.deepcopy(best_result_in_batch)

    # 현재 결과와 occupancy를 저장합니다.
    current_result = copy.deepcopy(best_result_in_batch)
    current_occupancies = current_result.orbital_occupancies

# 최소 한 번의 반복이 있었으므로 best_result는 None이 아닙니다.
best_result = cast(SCIResult, best_result)

In [ ]:
print(f"Best energy (reference-augmented SQD) = {best_result.energy + nuclear_repulsion_energy}")

plt.axhline(scf.e_tot, color="C0", label="HF")
plt.axhline(ccsd.e_tot, color="C1", label="CCSD")
plt.axhline(ref_result.energy + nuclear_repulsion_energy, color="C2", label="Reference Subspace")
plt.boxplot(
    [[x.energy + nuclear_repulsion_energy for results in result_history for x in results],
     [x.energy + nuclear_repulsion_energy for results in pm_result_history for x in results]],
    tick_labels=["Original SQD", "Reference-Augmented SQD"],
)
plt.legend(loc='upper right')
plt.show()

**Reference-Augmented SQD** 박스 그래프에서 두 가지 개선 사항을 확인하세요:

1. **더 낮은 에너지** — 최상의 측정값이 CCSD 기준에 더 가깝습니다. 100개의 고정된 reference determinant가 sampling 노이즈에 관계없이 가장 중요한 near-HF 구성들이 항상 부분공간에 포함되도록 보장하기 때문입니다.
2. **더 좁은 분포** — 박스 그래프의 분산이 적어진 것은, 결과가 배치 간에 더 *안정적*임을 의미합니다. Reference determinant로 부분공간을 고정하면 잘못된 샘플링으로 인한 분산이 줄어듭니다.

**Reference-Augmented SQD 에너지가 순수 고전 reference subspace 에너지보다 낮다면 — 축하합니다. 실제 양자 하드웨어에서 quantum utility를 시연한 것입니다.** 양자 샘플들이 brute-force 열거법으로는 찾을 수 없는 구성들을 기여했고, 그 덕분에 측정 가능하게 더 나은 바닥 상태 에너지 측정값을 얻었습니다. 이것이 실제 quantum utility의 모습입니다: 양자 컴퓨터가 고전적인 열거법으로는 찾을 수 없는 무언가를 추가한 것입니다. 잘 하셨습니다.

## Bonus Challenge: Further improve the energy estimation

### Not required for the badge

이제 직접 **quantum advantage** 를 향해 에너지 측정값을 더 밀어붙일 차례입니다!

<a id="exercise_bonus"></a>
<div class="alert alert-block alert-success">

<b>Bonus exercise: Find your best subspace!</b>

**목표:** N₂에 대한 SQD 에너지 측정값을 가능한 한 낮게 만드세요.

**제약 조건:** 부분공간 차원은 **90,000** 을 초과하면 안 됩니다(즉, `max_dim`을 최대 `(300, 300)`으로 유지하세요). 이 제한은 공정한 비교를 위해 모든 참가자에게 적용됩니다.

**힌트:**

- **전략적 부분공간 정제:** 현재 SQD 루프가 반복을 거쳐도 결과를 개선하지 못하는 것을 눈치챘을 것입니다. 품질 좋은 determinant 일부를 유지하고 SQD 반복 간에 carry-over하는 방식으로 부분공간을 더 전략적으로 정제해 보세요.

- **다른 구성 복원 방식:** 구성 복원에 사용되는 함수가 샘플 공간의 품질을 결정합니다. `prob_flip_0_to_1`에 대해 다른 함수 형태를 실험하여 복원된 구성들을 더 높은 품질의 샘플로 유도해 보세요.

- **더 좋은 ansatz:** LUCJ 레이어를 늘리거나 파라미터를 더 최적화하면 양자 샘플의 품질을 향상시킬 수 있습니다. 위의 보강된 방법에서 확인했듯이, 샘플 품질은 전체 SQD 성능과 수렴 속도에 매우 중요합니다.

- **더 많은 반복:** 준비가 됐다면, `max_iterations`나 `num_batches`를 늘려 SQD 루프가 구성을 더 많이 복원하도록 하세요.

**기대치:** 안타깝게도 주어진 부분공간 크기에서는 CCSD 결과를 능가하지 못할 수도 있습니다. 그러나 더 큰 부분공간 차원으로 확장하기 전에 SQD 알고리즘을 개선하는 중요한 기법들을 연습할 수 있습니다. 이 보너스 챌린지에서는 0에서 100까지 매겨지는 점수를 참고해보세요. 점수는 에너지 최소화 정도를 반영해 최대 100점까지 매겨집니다. 가능하면 100점을 목표로 해보세요!

아래 셀에서 최고의 결과를 제출할 수 있습니다.

</div>

In [ ]:
best_energy = best_result.energy + nuclear_repulsion_energy
best_subspace = (best_result.sci_state.ci_strs_a, best_result.sci_state.ci_strs_b)
grade_lab4c_exbonus(best_energy, best_subspace)  # 최고의 에너지와 부분공간을 제출합니다.

잘 하셨습니다 — 보너스 챌린지를 완료했습니다!

# Additional information

**Created by:** Boseong Kim

**Version:** 1.0.0